![Ecommerce Sales Cover](ecommerce_sales_cover.png)

# Sales Forecasting and Market-Basket Analysis for a Kenyan Baby Products Retailer

This project predicts future sales and uncovers which products are bought together, using 36 months of data from a Kenyan baby-products retailer. The results help the business plan stock and design bundles that increase order value.

### 1.Business Understanding

### 1.1 Background

The client is a Kenyan retailer of baby products that sells through three channels: a physical point of sale, an online store and draft orders. The dataset covers 36 months of trading, from November 2020 to October 2023, and contains 28,755 transaction lines recorded in Kenyan shillings (KES). Over this period the business grew quickly, with yearly order counts rising from about 3,000 in 2021 to over 5,700 in the first ten months of 2023, and monthly net sales climbing from under KES 1 million to a peak of roughly KES 4.4 million.

This growth makes planning harder. Stock, cash and promotions are still decided largely from experience, so the retailer risks running out of fast-moving items such as formula and diapers, or tying up money in slow-moving ones. The retailer also does not know which products customers tend to buy together, even though about two in three orders contain more than one distinct product.

In [1]:
# core libraries and notebook display settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)


In [ ]:
load_dotenv()

PGUSER = os.getenv("PGUSER")
PGPASSWORD = os.getenv("PGPASSWORD")
PGHOST = os.getenv("PGHOST")
PGPORT = os.getenv("PGPORT")
PGDATABASE = os.getenv("PGDATABASE")

# Build the connection string
DATABASE_URL = f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}?sslmode=require"

engine = create_engine(DATABASE_URL)
print("Connected to cloud database (Neon) — engine ready!")

In [8]:

load_dotenv(override=True)

neon_url = os.getenv("NEON_URL").replace("postgresql://", "postgresql+psycopg2://", 1)
engine = create_engine(neon_url)

df = pd.read_sql("SELECT * FROM sales", con=engine)
print(df.shape)   # expect (28755, 17)
df.head()

(28755, 17)


,sale_id,order_id,date,order_name,transaction_type,sale_type,sales_channel,product_type,product,net_quantity,gross_sales,discounts,returns,net_sales,shipping,taxes,total_sales
0,8725409595544,2854884835480,2020-11-10 22:39:33+00:00,TS1011,product,order,Online Store,Bathing & Skin Care,Mamia Sensitive Bathing & Skin Care 64 Bathing...,1,200.0,0.0,0.0,200.0,0.0,0.0,200.0
1,8752046604440,2864488448152,2020-11-16 13:43:01+00:00,TS1012,product,order,Point of Sale,Baby Formula,NANNYcare Growing Up Milk Goat Milk Based 3 Fr...,2,9000.0,0.0,0.0,9000.0,0.0,0.0,9000.0
2,8752055845016,2864491954328,2020-11-16 13:45:58+00:00,TS1013,product,order,Point of Sale,Baby Diapers,DryNites Spider-Man Baby & Toddler Clothing 16pk,1,1800.0,0.0,0.0,1800.0,0.0,0.0,1800.0
3,8752055877784,2864491954328,2020-11-16 13:45:58+00:00,TS1013,product,order,Point of Sale,Bathing & Skin Care,Metanium Nappy Rash Ointment 30g,1,1500.0,0.0,0.0,1500.0,0.0,0.0,1500.0
4,8752070656152,2864497066136,2020-11-16 13:49:47+00:00,TS1014,product,order,Point of Sale,Baby Formula,Aptamil 1 First Baby Milk Formula from Birth 800g,1,3000.0,0.0,0.0,3000.0,0.0,0.0,3000.0


In [11]:
SOURCE = "Local"    # change to "Neon" for the cloud database

if SOURCE == "Local":
    url = (f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
           f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}?sslmode=prefer")
else:
    url = os.getenv("NEON_URL").replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(url)
df = pd.read_sql("SELECT * FROM sales", con=engine)
print(SOURCE, df.shape)

Local (28755, 17)
